# Model

## Characteristics

- autocorrelated data
- time series data

## Tasks

- feature selection (w/ which technic?) (avoid data leakage of data that is not available when planning)
- choose model (only go for one approach, show why others were not pursued)
- avoid overfitting (how?)

## Objectives

- predict on a daily basis the amount of standby drivers efficiently (maximize rate of called in standby drivers)
- minimize days w/ too little drivers (&rarr; dafted drivers needed) (might result in very little days w/ way to many missing drivers, look out for that)
- -> exploitation rate of n_sby for days w/ sby_need/n_sby < 1 and amount of days w/ sby_need/n_sby > 1 

## Restrictions

- plan will be created on the 15th for the following month -> last half of month should not be included in training data

## Further Requirements

- discuss feature importance to increase trust in model (not applicable)
- make predictions as interpretable as possible
- detailed failure analysis to asses situations for which model is not suited
    - plot error in histogram (should be gauss if accumulation somewhere inspect those samples and look for commonalities)
- look up script for only using 90th percentile (for more robustness)

In [ ]:
# overall structure (_pred for predicted features)
# n_sby_pred = n_work_pred + n_sick_pred - n_duty (transformed from calculation of n_work above)
#   n_work_pred predicted by linear regression of calls_pred
#       calls_pred is predicted by time series prediction
#   n_sick_pred is predicted by time series prediction

# TODO:
# o save dataset which is used for training (maybe train in new notebook)
# o develop model
#   - SARIMA:
#       Pros: interpretable, provides confidence intervals, clear trend and seasonality
#       Cons: maybe less powerful, no external features, maybe seasonality to complex, only for smaller datasets, problem: yearly seasonality but daily output does not work
#   - XGBoost/NNs: Problem: need many lags (1, 2, 3, 7, 30, 365, ...) because FFT shows influence
#   - Prophet: additive model, high interpretability due to its decomposable components
#       o components are according to FFT (default: yearly and weekly)
#       o has plot_components() function for visualization (shows also change_points, can be changed by changepoint_prior_scale)
#       o start w/ default and look at residuals (sharp spikes: add_regressors(), smooth wave: add_seasonality())
#       o components can be added w/ add_seasonality(name='monthly', period=30.5, fourier_order=5)) (fourier_order: the higher the more flexible)
#       o regressors are extra features (like is_month_start)
#   o predict calls (for that only date and previous calls (trend, seasonal, residuals) are necessary (everything else is not related))

#   o prediction needs to be on a daily basis
#     (e.g. w/ lib: from statsforecast import StatsForecast)
#   o relevant columns: calls_trend, calls_seas, calls_resid, year, month, day, perc_sick_trend, perc_sick_seas, perc_sick_resid
#     (probably only calls and perc_sick needed (or are trend and seas helpful as well?))
# o possibilities to include days w/ too little n_sby and exploitation rate in penalty term?
#   (would make a retraining of submodels during training of overall model necessary)
#   (would work w/ simple model for calls_pred and n_sick_pred) (or just tweaking quantiles/hyperparameters of training)
# o point estimator (e.g. w/ mu and sigma (according distribution which is similar to distribution of feature itself (here Gauss I think))
#   can be trained on higher level? -> then with goals (exploitation and little days >100%)
#   use quantil in cost function? (p. 80)

# o avoid overfitting through trainings and testdata
# o use hyperparameter (determine w/ cross validation, brute force?, w/ 3rd dataset)
#   cross validation (usually w/ small datasets) w/ time series:
#   make sure that model sees all seasonalities in one training split (test also needs to be for all seasons)
#   fixed origin: 1. train, 2. test; 1.+2. train, 3.test; ... (1. = time interval)
#   rolling window: 1. train, 2. test; 2. train, 3.test; ... (test always smaller than train)
#   e.g. sklearn.model_selection.TimeSeriesSplit
# o penalty terms for reducing complexity here applicable?
# o check for structure in the residuals of the model (there should be none) (plot residual over time, in histogram, in probplot,
#   in correlogram (lags should not be correlated at all (only 1.)))
# o evaluation of model!
#   o metrics in lection 7 p. 64 and p. 77ff. (reliability diagram, investigating quantiles w/ PIT, pinball loss for enforcing defined quantiles)
#     MAPE might be good

# o make interpretable
#   o linear models/GLM (generalized linear models) show influence of features w/ coefficients
#   o (decision trees are completely interpretable but here not applicable)
#   o are there surrogate-models for the time series models which can approximate individual predictions (and therefore) explain them?

# o for restriction of needing time plan already on the 15th: just predict always 1.5 months